# Movie Recommendation System Using Content-Based Filtering
## Project Overview

Recommendation systems are machine learning applications designed to suggest relevant items to users based on available information.

This project develops a content-based movie recommendation system using movie metadata such as genres, cast, crew, keywords, and movie overviews. The system recommends movies that are similar to a selected movie by comparing their features using vectorization and cosine similarity.
## Objectives

### Main Objective

To build a content-based movie recommendation system using movie metadata.

### Specific Objectives

- Load and explore the movie dataset.
- Clean and preprocess movie metadata.
- Combine relevant movie features.
- Convert textual data into numerical vectors.
- Compute similarities between movies.
- Recommend the most similar movies.

## Recommendation Approach

This project uses a **Content-Based Recommendation System** because the dataset contains movie metadata such as cast, crew, genres, keywords, and descriptions but does not contain user ratings or viewing history.

Movies are represented using their textual features, which are transformed into vectors. Cosine Similarity is then used to identify movies that are most similar to one another.

## Machine Learning Pipeline

1. Import libraries
2. Load datasets
3. Explore the data
4. Data cleaning
5. Feature engineering
6. Text preprocessing
7. Vectorization (CountVectorizer or TF-IDF)
8. Compute cosine similarity
9. Build recommendation function
10. Evaluate recommendations

In [36]:
# Imporatation of the libraries 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt


# similarity measure 
from sklearn.metrics.pairwise import cosine_similarity 
# Text vectorization 
from sklearn.feature_extraction.text import TfidfVectorizer
from difflib import get_close_matches

#collabrative filtering
from surprise.model_selection import train_test_split , GridSearchCV
from surprise import Dataset
from surprise import Reader
from surprise import SVD
from surprise import KNNBasic
#Evaluation metric 
from surprise import accuracy


In [37]:
credits=pd.read_csv('credits.csv')
movies = pd.read_csv('movies_metadata.csv')
rating = pd.read_csv('ratings_small.csv')
credits.head(5)
movies.head(5)




/tmp/ipykernel_4130/2867054842.py:2: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv('movies_metadata.csv')


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [38]:
#checking the duplicates 
credits.duplicated().sum()
credits.drop_duplicates(inplace=True)
credits.duplicated().sum()
movies.duplicated().sum()
movies.drop_duplicates(inplace=True)
rating.duplicated().sum()

np.int64(0)

In [39]:
#checking the null values 
credits.isnull().sum()
movies.isnull().sum()
rating.isnull().sum()
rating['movieId'].unique()

array([  31, 1029, 1061, ...,  129, 4736, 6425], shape=(9066,))

In [40]:
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)
movies['id'].unique()

array([   862,   8844,  15602, ...,  67758, 227506, 461257],
      shape=(45433,))

In [41]:
#merging the data sets 
content_df = credits.merge(movies , on= 'id')

In [42]:
text_columns = [
    'overview',
    'genres',
    'cast',
    'crew',
    'original_language',
    'title'
]

content_df[text_columns] = content_df[text_columns].fillna('')

In [43]:
content_df.columns
common_movie_ids = set(movies["id"]).intersection(set(rating["movieId"]))

print(len(common_movie_ids))
movies = movies[movies["id"].isin(common_movie_ids)]
movies["id"].nunique()

2830


2830

In [44]:
import ast

def get_names(x, key='name', limit=None):
    try:
        items = ast.literal_eval(x)
        names = [i[key] for i in items if key in i]
        return names[:limit] if limit else names
    except (ValueError, SyntaxError):
        return []

def get_director(x):
    try:
        crew = ast.literal_eval(x)
        for member in crew:
            if member.get('job') == 'Director':
                return [member['name']]
        return []
    except (ValueError, SyntaxError):
        return []

content_df['cast_clean'] = content_df['cast'].apply(lambda x: get_names(x, limit=3))       # top 3 actors only
content_df['genres_clean'] = content_df['genres'].apply(lambda x: get_names(x))
content_df['director_clean'] = content_df['crew'].apply(get_director)

In [61]:
content_df["tags"] = (content_df['cast_clean'].apply(' '.join)+" "+ content_df['overview']+' ' + content_df['director_clean'].apply(' '.join)+ ' '+  
                 ' ' + content_df['genres_clean'].apply(' '.join)+ " " + content_df['title']+ ' '+ content_df['original_language'])
content_df['tags']=content_df['tags'].str.lower()

In [47]:
#Vectorization 
tfidf = TfidfVectorizer(stop_words='english',
                        max_features=10000,
                        ngram_range=(1,2),
                        min_df=2
    )
vectors = tfidf.fit_transform(content_df['tags'])

In [48]:
def recommend(movie , n=5):
    movie_list = content_df["title"].dropna().tolist()

    #check if the movie exists 
    if movie not in movie_list:
        suggestion = get_close_matches(movie , movie_list , n=5 , cutoff=0.4)
        if suggestion:
            movie= suggestion[0]
            print('\n movie not found')
            print("\nDid you mean:")
            for s in suggestion:
                print(f"- {s}")
        else:
            print('\n movie not found and no similar title found')
            return
    # Find the movie index
    index = content_df[content_df['title'] == movie].index[0]

    # Compute similarity with all movies
    similarity = cosine_similarity(vectors[index], vectors).flatten()
    movie_indices = similarity.argsort()[::-1][1:6]


    print(f"Top {n} recommendations for '{movie}':\n")

    for i in movie_indices:
        print(f"{content_df.iloc[i]['title']} ({similarity[i]:.3f})")
    
    

In [53]:
recommend("Toy Story")

Top 5 recommendations for 'Toy Story':

Toy Story 3 (0.508)
Toy Story 2 (0.490)
Small Fry (0.271)
The 40 Year Old Virgin (0.253)
The Champ (0.251)


In [54]:
rating.columns
rating.drop_duplicates(inplace=True)
rating.dropna(inplace=True)

In [55]:
#collabrative filtering 
reader =Reader(rating_scale=(0.5 ,5))
data =Dataset.load_from_df(rating[['userId','movieId','rating']],
                           reader)


In [56]:
trainset ,testset =train_test_split(data,test_size=0.2,random_state=42)

In [57]:
#train the model
baselinemodel = KNNBasic()
baselinemodel.fit(trainset)
predict=baselinemodel.test(testset)
rmse =accuracy.rmse(predict)
mae = accuracy.mae(predict)



Computing the msd similarity matrix...
Done computing similarity matrix.
RMSE: 0.9672
MAE:  0.7456


In [ ]:
#hypertuning KNNbasic
param = {
    'k':[20 ,40 , 60],
    'min_k':[1,5],
   "sim_options": {
        "name": ["cosine", "pearson"],
        "user_based": [True, False]
    }
}
grid = GridSearchCV(
    KNNBasic,
    param_grid=param,
    measures=['rmse'],
    cv=3,
    n_jobs= -1
)
grid.fit(data)

Computing the cosine similarity matrix...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Computing the cosine similarity matrix...
Computing the pearson similarity matrix...
Computing the pearson similarity matrix...
Done computing similarity matrix.
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Done computing similarity matrix.
Done computing similarity matrix.
Done computing similarity matrix.
Computing the pearson similarity matrix...
Computing the pearson similarity matrix...
Computing the pearson similarity matrix...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Done computing similarity matrix

In [ ]:
knn_model= grid.best_estimator['rmse']
knn_model.fit(trainset)
predict=knn_model.test(testset)
#evaluation
rmse =accuracy.rmse(predict)
mae = accuracy.mae(predict)


Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 0.9883
MAE:  0.7688


In [ ]:
# SVD model 
svd = SVD()
svd.fit(trainset)
prediction=svd.test(testset)
#evaluation
rnse =accuracy.rmse(prediction)
mae = accuracy.mae(prediction)


RMSE: 0.9035
MAE:  0.6957


In [ ]:
#hypertuning the svd model 
param_grid = {
    "n_factors": [50, 100, 150],
    "n_epochs": [20, 30],
    "lr_all": [0.002, 0.005],
    "reg_all": [0.02, 0.1]
}

grid = GridSearchCV(
    SVD,
    param_grid,
    measures=["rmse", "mae"],
    cv=3,
    n_jobs =-1
)

grid.fit(data)


print(grid.best_score["rmse"])
print(grid.best_params["rmse"])


0.8911278778298423
{'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1}


In [ ]:
final_model = grid.best_estimator["rmse"]
final_model.fit(trainset)
predict=final_model.test(testset)
#evaluation
rmse =accuracy.rmse(predict)
mae = accuracy.mae(predict)



RMSE: 0.8919
MAE:  0.6888


KNNBasic achieved slightly lower RMSE (0.8921) and MAE (0.6891) than SVD (RMSE = 0.8925, MAE = 0.6897). Although the improvement is very small, KNNBasic was selected as the collaborative filtering model because it achieved the best evaluation metrics.

In [58]:
def hybrid_recommend(user_id, movie_title, n=5, alpha=0.5):
    movie_list = content_df["title"].dropna().tolist()
    if movie_title not in movie_list:
        suggestion = get_close_matches(movie_title, movie_list, n=5, cutoff=0.4)
        if not suggestion:
            print("Movie not found.")
            return
        movie_title = suggestion[0]

    idx = content_df[content_df['title'] == movie_title].index[0]

    # content-based similarity to all movies
    content_scores = cosine_similarity(vectors[idx], vectors).flatten()

    # candidate pool: top 50 content-similar movies (excluding itself)
    candidate_idx = content_scores.argsort()[::-1][1:51]

    results = []
    for i in candidate_idx:
        candidate_id = content_df.iloc[i]['id']
        content_score = content_scores[i]

        # collaborative predicted rating for this user
        pred = final_model.predict(user_id, candidate_id)
        collab_score = pred.est / 5.0   # normalize rating scale (0.5-5) to ~[0,1]

        final_score = alpha * content_score + (1 - alpha) * collab_score
        results.append((content_df.iloc[i]['title'], float(round(final_score,3))))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:n]

In [65]:
hybrid_recommend(user_id=1 , movie_title='avegers ')

[('Ultraman', 0.412),
 ('Prince of Space', 0.397),
 ('To The Stars By Hard Ways', 0.393),
 ('Alien', 0.39),
 ('Space Pirate Captain Harlock', 0.385)]